# Understanding customer: Intent Classification (bike shop chatbot)
Architectures compared: **RNN, LSTM, BERT** (PyTorch + Hugging Face).

How to run (Colab): Runtime > Change runtime type > **GPU**, then upload `intents_train.csv` and `intents_test.csv` into a `data/` folder (or next to the notebook) and run all cells.

Outputs: `intents.csv` (test predictions), `model.pkl` (pickled model) and `intent_utils.py` (class definitions your peer needs to unpickle the model).

**Research journal** is built automatically in section 5, with one row per experiment.

In [6]:
!pip -q install transformers

## 1. Model code (saved to `intent_utils.py` so the pickle can be loaded elsewhere)

In [7]:
%%writefile intent_utils.py
import re
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

FALLBACK = 'Fallback Intent'


def simple_tokenize(text):
    return re.findall(r"[a-z0-9']+|[^\sa-z0-9]", str(text).lower())


class Vocab:
    def __init__(self, texts):
        counts = Counter(t for s in texts for t in simple_tokenize(s))
        self.itos = ['<pad>', '<unk>'] + sorted(counts)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode(self, text, max_len=32):
        ids = [self.stoi.get(t, 1) for t in simple_tokenize(text)][:max_len]
        if not ids:
            ids = [1]
        return ids + [0] * (max_len - len(ids))

    def __len__(self):
        return len(self.itos)


class RNNNet(nn.Module):
    # embedding -> (bi)RNN or (bi)LSTM -> mean pooling over real tokens -> linear
    def __init__(self, vocab_size, n_classes, cell='lstm', emb=64, hid=64, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        rnn_cls = nn.LSTM if cell == 'lstm' else nn.RNN
        self.rnn = rnn_cls(emb, hid, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hid * 2, n_classes)

    def forward(self, x):
        lengths = (x != 0).sum(1).clamp(min=1).cpu()
        packed = pack_padded_sequence(self.emb(x), lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        mask = (x != 0).unsqueeze(-1).float()[:, :out.size(1)]
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(self.drop(pooled))


class IntentClassifier:
    # picklable wrapper: predict(texts) -> list of intent names, with Fallback Intent below `threshold`
    def __init__(self, kind, model, classes, vocab=None, tokenizer=None, threshold=0.0, max_len=32):
        self.kind, self.model, self.classes = kind, model, list(classes)
        self.vocab, self.tokenizer = vocab, tokenizer
        self.threshold, self.max_len = threshold, max_len

    @torch.no_grad()
    def predict_proba(self, texts):
        texts = [str(t) for t in texts]
        self.model.cpu().eval()
        if self.kind == 'bert':
            enc = self.tokenizer(texts, padding=True, truncation=True,
                                 max_length=self.max_len, return_tensors='pt')
            logits = self.model(input_ids=enc['input_ids'],
                                attention_mask=enc['attention_mask']).logits
        else:
            x = torch.tensor([self.vocab.encode(t, self.max_len) for t in texts])
            logits = self.model(x)
        return torch.softmax(logits, dim=-1).numpy()

    def predict(self, texts, use_fallback=True):
        p = self.predict_proba(texts)
        idx, conf = p.argmax(1), p.max(1)
        return [self.classes[i] if (conf_i >= self.threshold or not use_fallback) else FALLBACK
                for i, conf_i in zip(idx, conf)]

Overwriting intent_utils.py


## 2. Imports and data

In [9]:
import os, pickle, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from intent_utils import Vocab, RNNNet, IntentClassifier, FALLBACK

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

def find(name):
    for d in ['data', '.', '/content', '/content/data']:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{name} not found - upload it to data/')

train_df = pd.read_csv(find('intents_train.csv'))
test_df  = pd.read_csv(find('intents_test.csv'))
print(train_df.shape, test_df.shape)
train_df.intent.value_counts()

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu
(94, 2) (37, 1)


intent
Return Policy       21
Bike Types          18
Make Appointment    16
Trade-in Options    16
Hours               15
Cost Estimation      7
Welcome Intent       1
Name: count, dtype: int64

## 3. Preprocessing and train / validation split
Preprocessing is deliberately light: strip whitespace, drop exact duplicates. Case is handled by each model's tokenizer (lowercasing for RNN/LSTM, `bert-base-uncased` for BERT). Many training phrases are short or cut off ("I got a ", "need to "), so stop-word removal would destroy signal.

`Welcome Intent` has only 1 example, so it can't be stratified. It always stays in train.

In [10]:
train_df['text'] = train_df.text.astype(str).str.strip()
test_df['text']  = test_df.text.astype(str).str.strip()
train_df = train_df.drop_duplicates('text').reset_index(drop=True)

CLASSES = sorted(train_df.intent.unique())
print(CLASSES)

single = train_df.groupby('intent').intent.transform('count') == 1
pool, fixed = train_df[~single], train_df[single]
tr, va = train_test_split(pool, test_size=0.2, stratify=pool.intent, random_state=SEED)
tr = pd.concat([tr, fixed]).reset_index(drop=True)
va = va.reset_index(drop=True)
print('train', len(tr), '| validation', len(va))

['Bike Types', 'Cost Estimation', 'Hours', 'Make Appointment', 'Return Policy', 'Trade-in Options', 'Welcome Intent']
train 75 | validation 19


## 4. Fallback Intent
The test file contains questions that belong to **no** known intent, and they count toward the score. Rule: if the model's top softmax probability is below a threshold, answer `Fallback Intent`.

The threshold is tuned on the validation split plus the out-of-scope phrases below (bike-shop-adjacent and unrelated questions I wrote; none come from the test file). Edit or extend the list for a better threshold.

In [11]:
OOD = [
    "What's the weather like today?", "Tell me a joke", "Who won the football game last night?",
    "Do you sell pizza?", "How do I reset my email password?", "What is the capital of France?",
    "Can you recommend a good restaurant?", "I want to book a flight to Paris",
    "Do you offer car insurance?", "How tall is Mount Everest?", "Play some music",
    "What's the meaning of life?", "My laptop screen is broken",
    "Do you sell bike locks?", "Do you offer discounts for students?",
    "Do you organize group rides?", "Where can I find bike trails nearby?",
    "Do you sell cycling clothing?", "Do you have a loyalty program?",
    "Who is the owner of the shop?", "Do you have parking?",
]
print(len(OOD), 'out-of-scope phrases')

21 out-of-scope phrases


## 5. Training function and research journal

In [12]:
MAX_LEN = 32
BERT_NAME = 'bert-base-uncased'

def fit(kind, texts, labels, cfg, seed=SEED):
    # kind: 'rnn' | 'lstm' | 'bert'. Returns an IntentClassifier (on CPU, threshold not set yet).
    torch.manual_seed(seed); np.random.seed(seed)
    texts = [str(t) for t in texts]
    y = torch.tensor([CLASSES.index(l) for l in labels])
    if kind == 'bert':
        tok = AutoTokenizer.from_pretrained(BERT_NAME)
        model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=len(CLASSES))
        enc = tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors='pt')
        ds = TensorDataset(enc['input_ids'], enc['attention_mask'], y)
        fwd = lambda b: model(input_ids=b[0], attention_mask=b[1]).logits
        clf = IntentClassifier('bert', model, CLASSES, tokenizer=tok, max_len=MAX_LEN)
    else:
        vocab = Vocab(texts)
        model = RNNNet(len(vocab), len(CLASSES), cell=kind, emb=cfg.get('emb', 64), hid=cfg.get('hid', 64))
        X = torch.tensor([vocab.encode(t, MAX_LEN) for t in texts])
        ds = TensorDataset(X, y)
        fwd = lambda b: model(b[0])
        clf = IntentClassifier(kind, model, CLASSES, vocab=vocab, max_len=MAX_LEN)

    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=0.01)
    dl = DataLoader(ds, batch_size=cfg.get('bs', 16), shuffle=True)
    for ep in range(cfg['epochs']):
        model.train()
        for b in dl:
            b = [t.to(DEVICE) for t in b]
            loss = F.cross_entropy(fwd(b), b[-1])
            opt.zero_grad(); loss.backward(); opt.step()
    model.cpu().eval()
    return clf


def tune_threshold(conf_val, correct_val, conf_ood):
    # threshold that best balances: in-scope answers kept AND correct, out-of-scope rejected
    grid = np.arange(0.30, 0.99, 0.01)
    scores = [0.5 * ((correct_val & (conf_val >= t)).mean() + (conf_ood < t).mean()) for t in grid]
    best = max(scores)
    ties = [t for t, s in zip(grid, scores) if s >= best - 1e-9]
    return float(np.median(ties)), best


journal, models = [], {}
yv = np.array([CLASSES.index(l) for l in va.intent])

def run_experiment(name, kind, cfg, notes=''):
    clf = fit(kind, tr.text, tr.intent, cfg)
    p_val, p_ood = clf.predict_proba(va.text), clf.predict_proba(OOD)
    pred, conf_val, conf_ood = p_val.argmax(1), p_val.max(1), p_ood.max(1)
    thr, bal = tune_threshold(conf_val, pred == yv, conf_ood)
    clf.threshold = thr
    models[name] = clf
    journal.append({
        'experiment': name, 'architecture': kind, 'framework': 'PyTorch' + (' + HF Transformers' if kind == 'bert' else ''),
        'config': str(cfg), 'preprocessing': 'strip, dedupe, lowercase tokens', 'train_size': len(tr),
        'val_acc': round(float((pred == yv).mean()), 3),
        'val_acc_with_fallback': round(float(((pred == yv) & (conf_val >= thr)).mean()), 3),
        'ood_rejected': round(float((conf_ood < thr).mean()), 3),
        'threshold': round(thr, 2), 'balanced_score': round(bal, 3), 'notes': notes})
    print(journal[-1])

In [13]:
EXPERIMENTS = [
    ('RNN-64',      'rnn',  dict(emb=64,  hid=64,  epochs=80, lr=3e-3), 'baseline bi-RNN'),
    ('RNN-128',     'rnn',  dict(emb=128, hid=128, epochs=80, lr=3e-3), 'wider'),
    ('LSTM-64',     'lstm', dict(emb=64,  hid=64,  epochs=80, lr=3e-3), 'bi-LSTM'),
    ('LSTM-128',    'lstm', dict(emb=128, hid=128, epochs=80, lr=3e-3), 'wider'),
    ('BERT-3e-5',   'bert', dict(epochs=10, lr=3e-5), 'bert-base-uncased full fine-tune'),
    ('BERT-5e-5',   'bert', dict(epochs=10, lr=5e-5), 'higher lr'),
]
for name, kind, cfg, notes in EXPERIMENTS:
    run_experiment(name, kind, cfg, notes)

{'experiment': 'RNN-64', 'architecture': 'rnn', 'framework': 'PyTorch', 'config': "{'emb': 64, 'hid': 64, 'epochs': 80, 'lr': 0.003}", 'preprocessing': 'strip, dedupe, lowercase tokens', 'train_size': 75, 'val_acc': 0.632, 'val_acc_with_fallback': 0.474, 'ood_rejected': 0.81, 'threshold': 0.98, 'balanced_score': np.float64(0.642), 'notes': 'baseline bi-RNN'}
{'experiment': 'RNN-128', 'architecture': 'rnn', 'framework': 'PyTorch', 'config': "{'emb': 128, 'hid': 128, 'epochs': 80, 'lr': 0.003}", 'preprocessing': 'strip, dedupe, lowercase tokens', 'train_size': 75, 'val_acc': 0.737, 'val_acc_with_fallback': 0.632, 'ood_rejected': 0.619, 'threshold': 0.97, 'balanced_score': np.float64(0.625), 'notes': 'wider'}
{'experiment': 'LSTM-64', 'architecture': 'lstm', 'framework': 'PyTorch', 'config': "{'emb': 64, 'hid': 64, 'epochs': 80, 'lr': 0.003}", 'preprocessing': 'strip, dedupe, lowercase tokens', 'train_size': 75, 'val_acc': 0.684, 'val_acc_with_fallback': 0.526, 'ood_rejected': 0.762, 'thr

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `h

{'experiment': 'BERT-3e-5', 'architecture': 'bert', 'framework': 'PyTorch + HF Transformers', 'config': "{'epochs': 10, 'lr': 3e-05}", 'preprocessing': 'strip, dedupe, lowercase tokens', 'train_size': 75, 'val_acc': 0.737, 'val_acc_with_fallback': 0.421, 'ood_rejected': 0.952, 'threshold': 0.53, 'balanced_score': np.float64(0.687), 'notes': 'bert-base-uncased full fine-tune'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 434.62it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

{'experiment': 'BERT-5e-5', 'architecture': 'bert', 'framework': 'PyTorch + HF Transformers', 'config': "{'epochs': 10, 'lr': 5e-05}", 'preprocessing': 'strip, dedupe, lowercase tokens', 'train_size': 75, 'val_acc': 0.737, 'val_acc_with_fallback': 0.684, 'ood_rejected': 0.667, 'threshold': 0.54, 'balanced_score': np.float64(0.675), 'notes': 'higher lr'}


In [14]:
journal_df = pd.DataFrame(journal)
journal_df.to_csv('research_journal.csv', index=False)
journal_df.drop(columns=['framework', 'preprocessing', 'train_size'])

,experiment,architecture,config,val_acc,val_acc_with_fallback,ood_rejected,threshold,balanced_score,notes
0,RNN-64,rnn,"{'emb': 64, 'hid': 64, 'epochs': 80, 'lr': 0.003}",0.632,0.474,0.810,0.98,0.642,baseline bi-RNN
1,RNN-128,rnn,"{'emb': 128, 'hid': 128, 'epochs': 80, 'lr': 0...",0.737,0.632,0.619,0.97,0.625,wider
2,LSTM-64,lstm,"{'emb': 64, 'hid': 64, 'epochs': 80, 'lr': 0.003}",0.684,0.526,0.762,0.98,0.644,bi-LSTM
3,LSTM-128,lstm,"{'emb': 128, 'hid': 128, 'epochs': 80, 'lr': 0...",0.684,0.526,0.667,0.97,0.596,wider
4,BERT-3e-5,bert,"{'epochs': 10, 'lr': 3e-05}",0.737,0.421,0.952,0.53,0.687,bert-base-uncased full fine-tune
5,BERT-5e-5,bert,"{'epochs': 10, 'lr': 5e-05}",0.737,0.684,0.667,0.54,0.675,higher lr


Caveat: the validation split is only ~19 phrases, so every accuracy step is ~5 points and the threshold is tuned on the same data. Treat small differences between models as noise. Add your own conclusions about which architecture worked best and why here.

In [15]:
best = journal_df.sort_values(['balanced_score', 'val_acc'], ascending=False).iloc[0]
print('Best experiment:', best.experiment)
best_model = models[best.experiment]

pred_val = best_model.predict(va.text, use_fallback=False)
print(classification_report(va.intent, pred_val, zero_division=0))

for name in ['hello', 'what time do you shut', 'can I send back my bike', 'tell me about the moon']:
    print(name, '->', best_model.predict([name])[0])

Best experiment: BERT-3e-5
                  precision    recall  f1-score   support

      Bike Types       0.50      0.75      0.60         4
 Cost Estimation       1.00      1.00      1.00         2
           Hours       1.00      0.67      0.80         3
Make Appointment       0.67      0.67      0.67         3
   Return Policy       1.00      1.00      1.00         4
Trade-in Options       0.50      0.33      0.40         3

        accuracy                           0.74        19
       macro avg       0.78      0.74      0.74        19
    weighted avg       0.76      0.74      0.74        19

hello -> Fallback Intent
what time do you shut -> Hours
can I send back my bike -> Fallback Intent
tell me about the moon -> Fallback Intent


## 6. Final model: retrain on train + validation, predict test, save
The retrained model reuses the configuration and fallback threshold of the best experiment.

In [16]:
best_cfg = next(c for n, k, c, _ in EXPERIMENTS if n == best.experiment)
all_df = pd.concat([tr, va]).reset_index(drop=True)

final = fit(best.architecture, all_df.text, all_df.intent, best_cfg)
final.threshold = float(best.threshold)

test_df['intent'] = final.predict(test_df.text)
test_df.to_csv('intents.csv', index=False)
print(test_df.intent.value_counts())
test_df

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 485.88it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

intent
Fallback Intent     18
Trade-in Options     7
Bike Types           4
Return Policy        4
Hours                3
Make Appointment     1
Name: count, dtype: int64


,text,intent
0,Do I need to replace my helmet?,Fallback Intent
1,My bike was stolen what should I do?,Fallback Intent
2,What types or choices of bikes do I have in yo...,Bike Types
3,Do you rent bikes?,Trade-in Options
4,I'm a student and I'm doing a project...,Fallback Intent
5,Can I bring my dog into your shop?,Trade-in Options
6,Do you match prices?,Fallback Intent
7,can I get an appointment tomorrow?,Make Appointment
8,How much to repair my bike?,Fallback Intent
9,How do I know what size tire/tube I need?,Fallback Intent


In [17]:
with open('model.pkl', 'wb') as f:
    pickle.dump(final, f)

# sanity check: the way your peer will load it
import importlib, intent_utils; importlib.reload(intent_utils)
with open('model.pkl', 'rb') as f:
    reloaded = pickle.load(f)
assert reloaded.predict(test_df.text) == list(test_df.intent), 'reloaded model differs!'
print('model.pkl OK ->', reloaded.kind, 'threshold', round(reloaded.threshold, 2))

model.pkl OK -> bert threshold 0.53


## Submission checklist
- Commit: this notebook, `intent_utils.py`, `model.pkl`, `intents.csv`, `research_journal.csv`, and the `data/` folder.
- Peer usage: `pickle.load(open('model.pkl','rb')).predict(list_of_phrases)`; it needs `intent_utils.py` importable and `torch` + `transformers` installed.
- Bonus ideas: add more phrases per intent to the training data, try a CNN, and push accuracy toward 0.873.